In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Read Gold layer
df = spark.table("gold.aapl_stock_data").dropna().toPandas()
df = df.sort_values("date")

In [0]:
# --- Features and target ---
features = ["open", "high", "low", "volume", "ma_7d", "ma_30d", "volatility_7d"]
target = "close"

X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # shuffle=False critical for time-series
)

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Create parent directory first, then set the experiment
experiment_path = "/financial-pipeline/stock-forecast"

# This creates the full path including parent dirs if they don't exist
mlflow.create_experiment(experiment_path)
mlflow.set_experiment(experiment_path)

In [0]:
# Point MLflow at the Databricks-managed tracking & registry server
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")  # <-- this is what resolves CONFIG_NOT_AVAILABLE

mlflow.set_experiment("/financial-pipeline/stock-forecast")

In [0]:
# --- MLflow experiment tracking ---
# This is the key Databricks-native feature
mlflow.set_experiment("/financial-pipeline/stock-forecast")

with mlflow.start_run():
    model = GradientBoostingRegressor(n_estimators=100, max_depth=4)
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    # Log parameters and metrics
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    
    # Log and register the model
    mlflow.sklearn.log_model(
        model,
        artifact_path="stock_forecast_model",
        registered_model_name="stock_forecast"
    )
    
    print(f"MAE: {mae:.4f} | R2: {r2:.4f}")